# Centered Tumor Crops

Takes the proportional-scale `32_seg/` data and shifts each tumor to the
center of the volume. The `novel_segmentation` diffusion model is trained
on these centered crops, so the model learns to generate a "centered tumor"
that the pipeline can then place at a realistic location at sampling time.

**Pipeline:** load 32_seg → find tumor centroid → shift to volume center
(nearest-neighbor, so the discrete labels {0,1,2,3} are preserved) → save.

**Input:**  `Processed/brats/32_seg/` (from `data_processing/processing_seg.ipynb`)

**Output:** `Processed/brats/32_seg_centered/`

**Optional CSV:** `tumor_size_labels.csv` (only used for the verification figure)


In [ ]:
from pathlib import Path

import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from tqdm.auto import tqdm


INPUT_DIR  = Path(r"C:\Users\Sven\Desktop\Data\Processed\brats\32_seg")
OUTPUT_DIR = Path(r"C:\Users\Sven\Desktop\Data\Processed\brats\32_seg_centered")
LABELS_CSV = Path.cwd() / "tumor_size_labels.csv"   # produced by analyze_tumor_size.ipynb


def center_tumor(volume, bg=0):
    # shift the tumor's center of mass to the middle of the volume.
    # nearest-neighbor (order=0) keeps the labels discrete; pad with bg.
    mask = volume > bg
    if not mask.any():
        return volume
    centroid = np.array(ndimage.center_of_mass(mask))
    shift = np.array(volume.shape) / 2.0 - centroid
    return ndimage.shift(volume, shift, order=0, mode="constant", cval=bg)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
input_files = sorted(INPUT_DIR.glob("*.nii.gz"))
print(f"input:  {INPUT_DIR}  ({len(input_files)} files)")
print(f"output: {OUTPUT_DIR}")

processed = skipped = 0
for f in tqdm(input_files, desc="centering tumors"):
    data = nib.load(str(f)).get_fdata()
    if (data > 0).sum() == 0:
        skipped += 1
        continue
    centered = center_tumor(data).astype(np.int16)
    nib.save(nib.Nifti1Image(centered, np.eye(4)), str(OUTPUT_DIR / f.name))
    processed += 1

print(f"processed {processed}, skipped {skipped} (no tumor)")


In [ ]:
output_files = sorted(OUTPUT_DIR.glob("*.nii.gz"))
label_df = pd.read_csv(LABELS_CSV).set_index("filename")

rng = np.random.default_rng(42)
show_files = rng.choice(output_files, size=min(4, len(output_files)), replace=False)

fig, axes = plt.subplots(len(show_files), 6, figsize=(18, 3 * len(show_files)))
views = ["axial", "coronal", "sagittal"]

for row, f in enumerate(show_files):
    orig = nib.load(str(INPUT_DIR / f.name)).get_fdata()
    cent = nib.load(str(f)).get_fdata()
    d = orig.shape[0] // 2

    slices_orig = [orig[d], orig[:, d], orig[:, :, d]]
    slices_cent = [cent[d], cent[:, d], cent[:, :, d]]

    for col, (img, name) in enumerate(zip(slices_orig, views)):
        axes[row, col].imshow(img, cmap="nipy_spectral", vmin=0, vmax=3)
        axes[row, col].set_title(f"original {name}" if row == 0 else "")
        axes[row, col].axis("off")
    for col, (img, name) in enumerate(zip(slices_cent, views), start=3):
        axes[row, col].imshow(img, cmap="nipy_spectral", vmin=0, vmax=3)
        axes[row, col].set_title(f"centered {name}" if row == 0 else "")
        axes[row, col].axis("off")

    if f.name in label_df.index:
        info = label_df.loc[f.name]
        sl = {0: "S", 1: "M", 2: "L"}[int(info["size_label"])]
        axes[row, 0].set_ylabel(f"{sl} | {info['volume_cm3']:.1f} cm³",
                                fontsize=11, rotation=0, labelpad=55, va="center")

plt.suptitle("original 32_seg vs centered 32_seg_centered", fontsize=14)
plt.tight_layout()
plt.show()
